# Practical 8 — Uber Fare Prediction

Standalone solution notebook for **ML Practical 8**. Run cells from top to bottom and place the required dataset CSV in the working directory.https://www.kaggle.com/datasets/yasserh/uber-fares-dataset/code/

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, mean_squared_error, mean_absolute_error,
    r2_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

---
# Practical 8 — Uber Fare Prediction
**Aim:** Preprocess Uber fare data, inspect outliers/correlation, train Linear Regression and Random Forest Regression models, and compare R², RMSE, and MAE.

Expected dataset: `uber.csv` or `Uber.csv` (Kaggle Uber Fares Dataset). Common columns include `fare_amount`, `pickup_longitude`, `pickup_latitude`, `dropoff_longitude`, `dropoff_latitude`, and `passenger_count`.

In [ ]:
# Change this path if your CSV is stored elsewhere.
UBER_PATH = "uber.csv"

if not os.path.exists(UBER_PATH):
    alternatives = ["Uber.csv", "uber_fares.csv", "uber_fares_dataset.csv"]
    found = next((f for f in alternatives if os.path.exists(f)), None)
    if found:
        UBER_PATH = found
    else:
        raise FileNotFoundError(
            "Uber dataset not found. Place the Kaggle CSV beside this notebook "
            "and set UBER_PATH to its filename."
        )

uber = pd.read_csv(UBER_PATH)
print("Shape:", uber.shape)
display(uber.head())
print(uber.info())

In [ ]:
# Standardize column names and select common fare dataset columns.
uber.columns = [c.strip().lower() for c in uber.columns]
required = [
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count"
]
missing = [c for c in required if c not in uber.columns]
if missing:
    raise ValueError(f"Required columns not found: {missing}. Available columns: {list(uber.columns)}")

uber = uber[required + (["pickup_datetime"] if "pickup_datetime" in uber.columns else [])].copy()
for c in required:
    uber[c] = pd.to_numeric(uber[c], errors="coerce")
uber = uber.replace([np.inf, -np.inf], np.nan).dropna(subset=required)

# Remove clearly invalid records and extreme fare values to reduce data-entry outliers.
uber = uber[
    (uber["fare_amount"] > 0) &
    (uber["passenger_count"].between(1, 6)) &
    (uber["pickup_latitude"].between(-90, 90)) &
    (uber["dropoff_latitude"].between(-90, 90)) &
    (uber["pickup_longitude"].between(-180, 180)) &
    (uber["dropoff_longitude"].between(-180, 180))
].copy()

# New feature: great-circle distance in kilometres (Haversine formula).
def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 2 * radius * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

uber["distance_km"] = haversine_km(
    uber["pickup_latitude"], uber["pickup_longitude"],
    uber["dropoff_latitude"], uber["dropoff_longitude"]
)
uber = uber[(uber["distance_km"] > 0) & (uber["distance_km"] < 100)].copy()

# Add date/time features when the datetime column is available.
if "pickup_datetime" in uber.columns:
    dt = pd.to_datetime(uber["pickup_datetime"], errors="coerce", utc=True)
    uber["pickup_hour"] = dt.dt.hour
    uber["pickup_day"] = dt.dt.dayofweek
    uber = uber.drop(columns=["pickup_datetime"])
    uber = uber.dropna(subset=["pickup_hour", "pickup_day"])

print("Cleaned shape:", uber.shape)
display(uber.describe().T)

In [ ]:
# Visualize fare outliers and feature correlation.
plt.figure(figsize=(8, 4))
sns.boxplot(x=uber["fare_amount"])
plt.title("Fare Amount — Box Plot")
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(uber.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = uber.drop(columns=["fare_amount"])
y = uber["fare_amount"]

# Keep the notebook responsive for very large CSV files; set SAMPLE_SIZE=None to use all rows.
SAMPLE_SIZE = 100_000
if SAMPLE_SIZE is not None and len(uber) > SAMPLE_SIZE:
    sampled = uber.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
    X, y = sampled.drop(columns=["fare_amount"]), sampled["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regression": RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
    )
}
regression_results = []
trained_regressors = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    regression_results.append({
        "Model": name,
        "R2": r2_score(y_test, pred),
        "RMSE": rmse,
        "MAE": mean_absolute_error(y_test, pred)
    })
    trained_regressors[name] = model

regression_results = pd.DataFrame(regression_results).sort_values("RMSE")
display(regression_results)

In [ ]:
best_regressor_name = regression_results.iloc[0]["Model"]
print("Best model by RMSE:", best_regressor_name)
best_regressor = trained_regressors[best_regressor_name]
sample_predictions = pd.DataFrame({
    "Actual Fare": y_test.iloc[:10].values,
    "Predicted Fare": best_regressor.predict(X_test.iloc[:10])
})
display(sample_predictions)

**Result:** The notebook cleans the Uber dataset, creates a Haversine distance feature, trains Linear Regression and Random Forest Regression, and compares R², RMSE, and MAE. Use the displayed results from your own run in your practical journal.